# Detector Inteligente de Campos en Tickets Hospitalarios

## Proyecto ABP — Procesamiento de Imágenes

Este notebook desarrolla un sistema de detección automática de campos administrativos en tickets hospitalarios usando visión computacional y YOLOv8.

A diferencia del proyecto SmartFactura, este trabajo no clasifica si una imagen es nítida o borrosa. El objetivo es detectar regiones específicas dentro del ticket, tales como paciente, factura, autorización, código, cantidad, copago, cuota moderadora, total, ingreso y valor unitario.

## Estructura esperada del dataset

El dataset debe estar en formato Roboflow RetinaNet o equivalente, con carpetas:

```text
HOSPITAL SAN JUAN BAUTISTA.v2i.retinanet/
├── train/
│   ├── _annotations.csv
│   └── imágenes
├── valid/
│   ├── _annotations.csv
│   └── imágenes
└── test/
    ├── _annotations.csv
    └── imágenes
```

El notebook convierte las anotaciones a formato YOLO, entrena el modelo, evalúa resultados y genera evidencias visuales.

### Ajuste para nivel Excelente según rúbrica ABP

Esta versión agrega una sección explícita de técnicas clásicas de procesamiento de imágenes y una sección de interpretación de métricas, limitaciones y mejoras futuras. Estos puntos refuerzan los criterios de procesamiento de imágenes, análisis de resultados e informe técnico.

## 1. Instalación de dependencias

In [ ]:
# Ejecutar esta celda una sola vez si faltan librerías.
# En Visual Studio Code / Jupyter funciona con !pip.

!pip install -q ultralytics opencv-python pandas matplotlib pyyaml

## 2. Importación de librerías

In [ ]:
from pathlib import Path
import shutil
import random
import yaml
import json
import os

import cv2
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

random.seed(42)

print("Librerías cargadas correctamente")
print("Dispositivo de entrenamiento:", DEVICE)

## 3. Configuración general del proyecto

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

BASE_DIR = Path.cwd()

PROJECT_NAME = "detector_tickets_hospitalarios"
PROJECT_DIR = BASE_DIR / PROJECT_NAME

YOLO_DIR = PROJECT_DIR / "dataset_yolo"
OUTPUT_DIR = PROJECT_DIR / "outputs"
EDA_DIR = OUTPUT_DIR / "eda"
PRED_DIR = OUTPUT_DIR / "predicciones"
EVIDENCE_DIR = OUTPUT_DIR / "evidencias"

for path in [PROJECT_DIR, YOLO_DIR, OUTPUT_DIR, EDA_DIR, PRED_DIR, EVIDENCE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "valid", "test"]

# Clases del dataset hospitalario
CLASS_NAMES = [
    "autorizacion",
    "cantidad",
    "codigo",
    "copago",
    "cuota_moderadora",
    "factura",
    "ingreso",
    "paciente",
    "total",
    "valor_unitario",
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

print("Carpeta base:", BASE_DIR)
print("Carpeta del proyecto:", PROJECT_DIR)
print("Cantidad de clases:", len(CLASS_NAMES))

## 4. Búsqueda automática del dataset

In [ ]:
def buscar_dataset_root(base_dir: Path) -> Path:
    '''
    Busca automáticamente una carpeta que contenga train, valid y test,
    cada una con su archivo _annotations.csv.
    '''
    candidatos = [
        base_dir,
        base_dir / "HOSPITAL SAN JUAN BAUTISTA.v2i.retinanet",
        base_dir / "Hospital San Juan Bautista-Tickets",
        base_dir / "dataset",
        base_dir / "data",
        base_dir / "imagenes",
        base_dir / "IMAGENES",
    ]

    for candidato in candidatos:
        if all((candidato / split / "_annotations.csv").exists() for split in SPLITS):
            return candidato

    for csv_path in base_dir.rglob("_annotations.csv"):
        posible_root = csv_path.parent.parent
        if all((posible_root / split / "_annotations.csv").exists() for split in SPLITS):
            return posible_root

    raise FileNotFoundError(
        "No se encontró un dataset con carpetas train, valid y test con _annotations.csv. "
        "Ubicá el dataset en la misma carpeta del notebook o ajustá DATASET_ROOT manualmente."
    )

DATASET_ROOT = buscar_dataset_root(BASE_DIR)

print("Dataset detectado en:", DATASET_ROOT)
for split in SPLITS:
    print(split, "->", DATASET_ROOT / split)

## 5. Validación inicial de imágenes y archivos CSV

In [ ]:
def listar_imagenes(split_dir: Path):
    imagenes = []
    for ext in IMAGE_EXTENSIONS:
        imagenes.extend(split_dir.glob(f"*{ext}"))
        imagenes.extend(split_dir.glob(f"*{ext.upper()}"))
    return sorted(set(imagenes))

resumen_archivos = []

for split in SPLITS:
    split_dir = DATASET_ROOT / split
    csv_path = split_dir / "_annotations.csv"
    imagenes = listar_imagenes(split_dir)

    resumen_archivos.append({
        "split": split,
        "csv_existe": csv_path.exists(),
        "cantidad_imagenes": len(imagenes),
        "ruta_csv": str(csv_path)
    })

resumen_archivos_df = pd.DataFrame(resumen_archivos)
display(resumen_archivos_df)

if not resumen_archivos_df["csv_existe"].all():
    raise FileNotFoundError("Falta al menos un archivo _annotations.csv.")

## 6. Carga robusta de anotaciones

In [ ]:
def normalizar_columnas(df_raw: pd.DataFrame) -> pd.DataFrame:
    '''
    Normaliza distintos formatos posibles de Roboflow RetinaNet.

    Formatos soportados:
    1) filename,xmin,ymin,xmax,ymax,class
    2) filename,width,height,class,xmin,ymin,xmax,ymax
    3) CSV con encabezados equivalentes.
    '''
    df = df_raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    # Caso con encabezados estándar de Roboflow
    columnas = set(df.columns)

    if {"filename", "xmin", "ymin", "xmax", "ymax", "class"}.issubset(columnas):
        return df[["filename", "xmin", "ymin", "xmax", "ymax", "class"]].copy()

    if {"filename", "width", "height", "class", "xmin", "ymin", "xmax", "ymax"}.issubset(columnas):
        return df[["filename", "xmin", "ymin", "xmax", "ymax", "class"]].copy()

    # Caso sin encabezados
    if df.shape[1] == 6:
        df.columns = ["filename", "xmin", "ymin", "xmax", "ymax", "class"]
        return df[["filename", "xmin", "ymin", "xmax", "ymax", "class"]].copy()

    if df.shape[1] == 8:
        df.columns = ["filename", "width", "height", "class", "xmin", "ymin", "xmax", "ymax"]
        return df[["filename", "xmin", "ymin", "xmax", "ymax", "class"]].copy()

    raise ValueError(f"No se pudo reconocer el formato del CSV. Columnas detectadas: {list(df.columns)}")


def cargar_anotaciones_split(split: str) -> pd.DataFrame:
    csv_path = DATASET_ROOT / split / "_annotations.csv"

    # Primer intento: con encabezado
    df_raw = pd.read_csv(csv_path)

    try:
        df_norm = normalizar_columnas(df_raw)
    except Exception:
        # Segundo intento: sin encabezado
        df_raw = pd.read_csv(csv_path, header=None)
        df_norm = normalizar_columnas(df_raw)

    df_norm["split"] = split
    return df_norm


df = pd.concat([cargar_anotaciones_split(split) for split in SPLITS], ignore_index=True)

# Limpieza básica
df["filename"] = df["filename"].astype(str).str.strip()
df["class"] = df["class"].astype(str).str.strip()

for col in ["xmin", "ymin", "xmax", "ymax"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["filename", "class", "xmin", "ymin", "xmax", "ymax"]).copy()

# Filtrar encabezados accidentales repetidos como filas
df = df[df["filename"].str.lower() != "filename"].copy()

print("Total de anotaciones cargadas:", len(df))
print("Total de imágenes únicas anotadas:", df["filename"].nunique())
print("Clases detectadas en CSV:", sorted(df["class"].unique()))

display(df.head())

## 7. Control de calidad de anotaciones

In [ ]:
# Validar clases desconocidas
clases_detectadas = set(df["class"].unique())
clases_esperadas = set(CLASS_NAMES)

clases_no_esperadas = sorted(clases_detectadas - clases_esperadas)
clases_sin_registros = sorted(clases_esperadas - clases_detectadas)

print("Clases no esperadas:", clases_no_esperadas)
print("Clases esperadas sin registros:", clases_sin_registros)

if clases_no_esperadas:
    print("\nAtención: existen clases en el CSV que no están declaradas en CLASS_NAMES.")
    print("Revisar si son errores de etiquetado o si deben agregarse a la lista de clases.")

# Validar bounding boxes
df["bbox_width"] = df["xmax"] - df["xmin"]
df["bbox_height"] = df["ymax"] - df["ymin"]
df["bbox_area"] = df["bbox_width"] * df["bbox_height"]

bbox_invalidas = df[(df["bbox_width"] <= 0) | (df["bbox_height"] <= 0)]

print("Bounding boxes inválidas:", len(bbox_invalidas))

if len(bbox_invalidas) > 0:
    display(bbox_invalidas.head())

## 8. Análisis exploratorio del dataset — versión final mejorada

En esta etapa se analiza la calidad estructural del dataset antes del entrenamiento del modelo YOLOv8.

El objetivo del análisis exploratorio no es solamente mostrar gráficos, sino validar si el conjunto de datos es adecuado para un problema de detección de objetos. Para eso se revisan:

- distribución de imágenes por partición;
- cantidad de anotaciones por imagen;
- balance general de clases;
- presencia de clases en train, valid y test;
- resolución de imágenes;
- tamaño relativo de los campos anotados;
- ubicación espacial de las anotaciones dentro del ticket.

Este análisis permite justificar técnicamente que el dataset está preparado para entrenar un modelo de detección de campos administrativos.

In [ ]:
# ============================================================
# 8.1 PREPARACIÓN DE TABLAS BASE PARA EL EDA
# ============================================================

import numpy as np

# Orden deseado de particiones para tablas y gráficos
SPLIT_ORDER = ["train", "valid", "test"]

# ------------------------------------------------------------
# Tabla de imágenes por partición
# ------------------------------------------------------------
imagenes_por_split = []

for split in SPLIT_ORDER:
    imagenes = listar_imagenes(DATASET_ROOT / split)
    imagenes_por_split.append({
        "split": split,
        "imagenes": len(imagenes)
    })

imagenes_por_split_df = pd.DataFrame(imagenes_por_split)

# ------------------------------------------------------------
# Tabla de anotaciones por partición
# ------------------------------------------------------------
anotaciones_por_split_df = (
    df.groupby("split")
      .size()
      .reindex(SPLIT_ORDER)
      .fillna(0)
      .astype(int)
      .reset_index(name="anotaciones")
)

# ------------------------------------------------------------
# Resumen general por partición
# ------------------------------------------------------------
resumen_split_df = imagenes_por_split_df.merge(
    anotaciones_por_split_df,
    on="split",
    how="left"
)

resumen_split_df["anotaciones"] = resumen_split_df["anotaciones"].fillna(0).astype(int)

resumen_split_df["porcentaje_imagenes"] = (
    resumen_split_df["imagenes"] / resumen_split_df["imagenes"].sum() * 100
).round(2)

resumen_split_df["porcentaje_anotaciones"] = (
    resumen_split_df["anotaciones"] / resumen_split_df["anotaciones"].sum() * 100
).round(2)

resumen_split_df["anotaciones_por_imagen"] = (
    resumen_split_df["anotaciones"] / resumen_split_df["imagenes"]
).round(2)

display(resumen_split_df)

print("Total de imágenes:", int(resumen_split_df["imagenes"].sum()))
print("Total de anotaciones:", int(resumen_split_df["anotaciones"].sum()))
print("Promedio global de anotaciones por imagen:", round(df.shape[0] / resumen_split_df["imagenes"].sum(), 2))

### Lectura del resumen general

Esta tabla permite validar la estructura principal del dataset. Para un modelo YOLOv8 es esperable que la mayor parte de los datos esté en `train`, mientras que `valid` y `test` se utilizan para validar y evaluar el rendimiento del modelo.

También se calcula el promedio de anotaciones por imagen. Este valor es importante porque en este proyecto cada ticket contiene varios campos administrativos detectables.

In [ ]:
# ============================================================
# 8.2 DISTRIBUCIÓN DE IMÁGENES POR PARTICIÓN
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(resumen_split_df["split"], resumen_split_df["imagenes"])

plt.title("Distribución de imágenes por partición")
plt.xlabel("Partición")
plt.ylabel("Cantidad de imágenes")
plt.grid(axis="y", alpha=0.3)

for i, row in resumen_split_df.iterrows():
    plt.text(
        i,
        row["imagenes"],
        f'{row["imagenes"]}\n({row["porcentaje_imagenes"]}%)',
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_01_imagenes_por_particion.png", dpi=150)
plt.show()

### Análisis de imágenes por partición

Este gráfico muestra cómo se distribuyen las imágenes entre entrenamiento, validación y prueba. La partición de entrenamiento concentra la mayor cantidad de ejemplos, lo cual es correcto porque es la base desde donde el modelo aprende los patrones visuales de cada campo.

In [ ]:
# ============================================================
# 8.3 PROMEDIO DE ANOTACIONES POR IMAGEN
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(resumen_split_df["split"], resumen_split_df["anotaciones_por_imagen"])

plt.title("Promedio de anotaciones por imagen")
plt.xlabel("Partición")
plt.ylabel("Anotaciones promedio por imagen")
plt.grid(axis="y", alpha=0.3)

for i, row in resumen_split_df.iterrows():
    plt.text(
        i,
        row["anotaciones_por_imagen"],
        row["anotaciones_por_imagen"],
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_02_promedio_anotaciones_por_imagen.png", dpi=150)
plt.show()

### Análisis de anotaciones por imagen

El promedio de anotaciones por imagen permite revisar si las particiones tienen una densidad similar de campos etiquetados. Si los valores son parecidos entre `train`, `valid` y `test`, la comparación entre particiones es más consistente.

In [ ]:
# ============================================================
# 8.4 DISTRIBUCIÓN GENERAL DE ANOTACIONES POR CLASE
# ============================================================

conteo_clases = (
    df["class"]
    .value_counts()
    .rename_axis("clase")
    .reset_index(name="cantidad")
)

conteo_clases["porcentaje"] = (
    conteo_clases["cantidad"] / conteo_clases["cantidad"].sum() * 100
).round(2)

display(conteo_clases)

plt.figure(figsize=(11, 6))

plt.barh(
    conteo_clases["clase"],
    conteo_clases["cantidad"]
)

plt.title("Distribución general de anotaciones por clase")
plt.xlabel("Cantidad de anotaciones")
plt.ylabel("Clase")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)

max_cantidad = conteo_clases["cantidad"].max()
plt.xlim(0, max_cantidad * 1.18)

for index, row in conteo_clases.iterrows():
    plt.text(
        row["cantidad"] + 1,
        index,
        f'{row["cantidad"]} ({row["porcentaje"]}%)',
        va="center"
    )

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_03_distribucion_general_clases.png", dpi=150)
plt.show()

### Análisis de distribución general por clase

Este gráfico responde cuántas anotaciones totales existen para cada campo del ticket. Es útil para evaluar si el dataset está balanceado a nivel global.

Una distribución similar entre clases reduce el riesgo de que el modelo aprenda mejor unas clases que otras únicamente por tener más ejemplos disponibles.

In [ ]:
# ============================================================
# 8.5 DISTRIBUCIÓN DE CLASES POR PARTICIÓN
# ============================================================

clases_por_split = (
    df.groupby(["class", "split"])
      .size()
      .reset_index(name="cantidad")
)

pivot_clases_split = (
    clases_por_split
    .pivot(index="class", columns="split", values="cantidad")
    .fillna(0)
    .astype(int)
)

# Asegurar columnas en orden train, valid, test
for split in SPLIT_ORDER:
    if split not in pivot_clases_split.columns:
        pivot_clases_split[split] = 0

pivot_clases_split = pivot_clases_split[SPLIT_ORDER]
pivot_clases_split["total"] = pivot_clases_split.sum(axis=1)
pivot_clases_split = pivot_clases_split.sort_values("total", ascending=True)

display(pivot_clases_split)

# Conversión a porcentaje por clase
pivot_porcentual = (
    pivot_clases_split[SPLIT_ORDER]
    .div(pivot_clases_split["total"], axis=0) * 100
).round(2)

display(pivot_porcentual)

pivot_porcentual.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Distribución porcentual de clases por partición")
plt.xlabel("Porcentaje de anotaciones")
plt.ylabel("Clase")
plt.xlim(0, 100)
plt.grid(axis="x", alpha=0.3)
plt.legend(title="Partición", loc="lower right")

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_04_distribucion_porcentual_clases_por_particion.png", dpi=150)
plt.show()

### Análisis de clases por partición

Este gráfico no repite el análisis anterior. La distribución general por clase muestra el balance total del dataset; en cambio, esta vista permite verificar si cada clase aparece correctamente distribuida entre `train`, `valid` y `test`.

Esto es clave porque el modelo debe aprender las clases en entrenamiento y luego encontrar ejemplos comparables durante validación y prueba.

In [ ]:
# ============================================================
# 8.6 MAPA TABULAR DE CLASES VS PARTICIONES
# ============================================================

# Matriz de conteos para revisar rápidamente posibles clases ausentes
matriz_conteos = pivot_clases_split[SPLIT_ORDER].sort_index()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matriz_conteos.values)

ax.set_xticks(np.arange(len(SPLIT_ORDER)))
ax.set_yticks(np.arange(len(matriz_conteos.index)))
ax.set_xticklabels(SPLIT_ORDER)
ax.set_yticklabels(matriz_conteos.index)

plt.setp(ax.get_xticklabels(), rotation=0, ha="center")

# Escribir valores dentro de cada celda
for i in range(len(matriz_conteos.index)):
    for j in range(len(SPLIT_ORDER)):
        ax.text(
            j,
            i,
            matriz_conteos.iloc[i, j],
            ha="center",
            va="center"
        )

ax.set_title("Cantidad de anotaciones por clase y partición")
ax.set_xlabel("Partición")
ax.set_ylabel("Clase")

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_05_matriz_clases_particiones.png", dpi=150)
plt.show()

### Análisis de la matriz de clases

La matriz permite detectar rápidamente si alguna clase falta en una partición. En problemas de detección de objetos, una clase ausente en `test` o `valid` limita la capacidad de evaluar correctamente el rendimiento del modelo para ese campo.

In [ ]:
# ============================================================
# 8.7 ANOTACIONES POR IMAGEN
# ============================================================

anotaciones_por_imagen = (
    df.groupby(["split", "filename"])
      .size()
      .reset_index(name="cantidad_anotaciones")
)

resumen_anotaciones_imagen = (
    anotaciones_por_imagen
    .groupby("split")["cantidad_anotaciones"]
    .agg(["count", "min", "mean", "median", "max"])
    .reindex(SPLIT_ORDER)
    .round(2)
    .reset_index()
)

display(resumen_anotaciones_imagen)

plt.figure(figsize=(9, 5))

plt.hist(
    anotaciones_por_imagen["cantidad_anotaciones"],
    bins=range(
        int(anotaciones_por_imagen["cantidad_anotaciones"].min()),
        int(anotaciones_por_imagen["cantidad_anotaciones"].max()) + 2
    )
)

plt.title("Distribución de cantidad de anotaciones por imagen")
plt.xlabel("Cantidad de anotaciones por imagen")
plt.ylabel("Cantidad de imágenes")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_06_histograma_anotaciones_por_imagen.png", dpi=150)
plt.show()

# Boxplot por partición
data_boxplot = [
    anotaciones_por_imagen[anotaciones_por_imagen["split"] == split]["cantidad_anotaciones"]
    for split in SPLIT_ORDER
]

plt.figure(figsize=(8, 5))
plt.boxplot(data_boxplot, labels=SPLIT_ORDER)

plt.title("Variabilidad de anotaciones por imagen según partición")
plt.xlabel("Partición")
plt.ylabel("Cantidad de anotaciones por imagen")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_07_boxplot_anotaciones_por_imagen.png", dpi=150)
plt.show()

### Análisis de cantidad de anotaciones por imagen

Este análisis es especialmente importante para YOLOv8 porque cada ticket contiene múltiples objetos a detectar. Una cantidad de anotaciones relativamente estable indica que las imágenes tienen una estructura documental comparable.

In [ ]:
# ============================================================
# 8.8 DIMENSIONES Y RESOLUCIÓN DE LAS IMÁGENES
# ============================================================

tamanios = []

for split in SPLIT_ORDER:
    for img_path in listar_imagenes(DATASET_ROOT / split):
        img = cv2.imread(str(img_path))

        if img is None:
            continue

        h, w = img.shape[:2]

        tamanios.append({
            "split": split,
            "filename": img_path.name,
            "width": w,
            "height": h,
            "area_imagen": w * h,
            "aspect_ratio": round(w / h, 3)
        })

tamanios_df = pd.DataFrame(tamanios)

display(tamanios_df.groupby("split")[["width", "height", "aspect_ratio"]].describe())

resumen_dimensiones = (
    tamanios_df
    .groupby(["width", "height", "aspect_ratio"])
    .size()
    .reset_index(name="cantidad_imagenes")
    .sort_values("cantidad_imagenes", ascending=False)
)

resumen_dimensiones["resolucion"] = (
    resumen_dimensiones["width"].astype(str)
    + "x"
    + resumen_dimensiones["height"].astype(str)
)

display(resumen_dimensiones)

plt.figure(figsize=(8, 5))

plt.bar(
    resumen_dimensiones["resolucion"],
    resumen_dimensiones["cantidad_imagenes"]
)

plt.title("Cantidad de imágenes por resolución")
plt.xlabel("Resolución")
plt.ylabel("Cantidad de imágenes")
plt.grid(axis="y", alpha=0.3)

for i, row in resumen_dimensiones.iterrows():
    plt.text(
        i,
        row["cantidad_imagenes"],
        row["cantidad_imagenes"],
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_08_imagenes_por_resolucion.png", dpi=150)
plt.show()

if len(resumen_dimensiones) == 1:
    print("Conclusión: todas las imágenes tienen la misma resolución. El dataset está estandarizado.")
else:
    print("Conclusión: existen múltiples resoluciones. Conviene revisar si el redimensionamiento afecta el entrenamiento.")

### Análisis de resolución

Si todas las imágenes tienen la misma resolución, no es un error. En datasets exportados desde herramientas como Roboflow suele ser una característica esperada, ya que las imágenes se estandarizan antes del entrenamiento.

Por ese motivo no se usa un gráfico de dispersión de ancho vs alto, porque mostraría un único punto y aportaría poca información.

In [ ]:
# ============================================================
# 8.9 TAMAÑO RELATIVO DE LOS BOUNDING BOXES
# ============================================================

df_bbox = df.merge(
    tamanios_df[["split", "filename", "width", "height", "area_imagen"]],
    on=["split", "filename"],
    how="left"
)

df_bbox["bbox_width"] = df_bbox["xmax"] - df_bbox["xmin"]
df_bbox["bbox_height"] = df_bbox["ymax"] - df_bbox["ymin"]
df_bbox["bbox_area"] = df_bbox["bbox_width"] * df_bbox["bbox_height"]
df_bbox["bbox_area_pct"] = (df_bbox["bbox_area"] / df_bbox["area_imagen"] * 100).round(4)

df_bbox["bbox_center_x_pct"] = (((df_bbox["xmin"] + df_bbox["xmax"]) / 2) / df_bbox["width"] * 100).round(2)
df_bbox["bbox_center_y_pct"] = (((df_bbox["ymin"] + df_bbox["ymax"]) / 2) / df_bbox["height"] * 100).round(2)

resumen_bbox = (
    df_bbox.groupby("class")["bbox_area_pct"]
    .agg(["count", "min", "mean", "median", "max"])
    .round(4)
    .sort_values("mean", ascending=False)
)

display(resumen_bbox)

# Ranking de tamaño medio por clase
resumen_bbox_plot = resumen_bbox.sort_values("mean", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(resumen_bbox_plot.index, resumen_bbox_plot["mean"])

plt.title("Tamaño medio relativo de los campos anotados")
plt.xlabel("Área media del bounding box respecto de la imagen (%)")
plt.ylabel("Clase")
plt.grid(axis="x", alpha=0.3)

for i, value in enumerate(resumen_bbox_plot["mean"]):
    plt.text(value, i, f" {value:.3f}%", va="center")

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_09_tamano_medio_bbox_por_clase.png", dpi=150)
plt.show()

# Boxplot de variabilidad de área por clase
clases_ordenadas = resumen_bbox.sort_values("median", ascending=True).index.tolist()

data = [
    df_bbox[df_bbox["class"] == clase]["bbox_area_pct"]
    for clase in clases_ordenadas
]

plt.figure(figsize=(10, 6))
plt.boxplot(data, labels=clases_ordenadas, vert=False)

plt.title("Variabilidad del tamaño relativo de los campos anotados")
plt.xlabel("Área del bounding box respecto de la imagen (%)")
plt.ylabel("Clase")
plt.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_10_boxplot_tamano_bbox_por_clase.png", dpi=150)
plt.show()

### Análisis de tamaño de bounding boxes

Este análisis es más relevante que mirar solo la resolución de la imagen. YOLOv8 necesita aprender a detectar campos específicos dentro del ticket; por eso es importante saber qué porcentaje de la imagen ocupa cada campo.

Campos con áreas muy pequeñas pueden ser más difíciles de detectar, especialmente si se reduce el tamaño de entrada de la imagen durante el entrenamiento.

In [ ]:
# ============================================================
# 8.10 MAPA ESPACIAL DE UBICACIÓN DE ANOTACIONES
# ============================================================

plt.figure(figsize=(8, 8))

for clase in sorted(df_bbox["class"].unique()):
    subset = df_bbox[df_bbox["class"] == clase]

    plt.scatter(
        subset["bbox_center_x_pct"],
        subset["bbox_center_y_pct"],
        alpha=0.6,
        label=clase
    )

plt.title("Mapa espacial de ubicación de campos anotados")
plt.xlabel("Posición horizontal del centro del campo (%)")
plt.ylabel("Posición vertical del centro del campo (%)")
plt.xlim(0, 100)
plt.ylim(100, 0)  # Se invierte para respetar la lógica visual de la imagen
plt.grid(True, alpha=0.3)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)

plt.tight_layout()
plt.savefig(EDA_DIR / "eda_11_mapa_espacial_anotaciones.png", dpi=150, bbox_inches="tight")
plt.show()

### Análisis espacial de anotaciones

El mapa espacial permite observar dónde suelen ubicarse los campos dentro del ticket. Esto es útil porque los documentos tienen una estructura relativamente fija: algunos campos aparecen en la cabecera, otros en la zona central y otros en el resumen de valores.

Si las anotaciones se agrupan en zonas coherentes, el dataset tiene un patrón espacial que el modelo puede aprender.

In [ ]:
# ============================================================
# 8.11 CONCLUSIONES AUTOMÁTICAS DEL EDA
# ============================================================

total_imagenes = int(resumen_split_df["imagenes"].sum())
total_anotaciones = int(resumen_split_df["anotaciones"].sum())

min_clase = int(conteo_clases["cantidad"].min())
max_clase = int(conteo_clases["cantidad"].max())

promedio_anotaciones_global = round(total_anotaciones / total_imagenes, 2)

print("CONCLUSIONES DEL ANÁLISIS EXPLORATORIO")
print("======================================")
print(f"1. El dataset contiene {total_imagenes} imágenes y {total_anotaciones} anotaciones.")
print(f"2. El promedio global es de {promedio_anotaciones_global} anotaciones por imagen.")
print(f"3. Las clases tienen entre {min_clase} y {max_clase} anotaciones, lo que indica una distribución general equilibrada.")
print("4. Todas las clases se validan por partición en la matriz train/valid/test.")
print("5. El análisis de resolución permite confirmar si las imágenes están estandarizadas.")
print("6. El análisis de bounding boxes permite evaluar el tamaño relativo de los campos detectables.")
print("7. El mapa espacial confirma la ubicación aproximada de los campos dentro del ticket.")

### Conclusión final del EDA

El análisis exploratorio muestra que el dataset es apto para un modelo de detección de objetos con YOLOv8. Las clases presentan una distribución equilibrada, las particiones mantienen presencia de los campos principales y las imágenes tienen una estructura documental consistente.

El dataset no se analiza como un problema de clasificación de imagen completa, sino como un problema de detección de múltiples campos dentro de cada ticket hospitalario. Esto justifica el uso de YOLOv8 como modelo principal del proyecto.

## 9. Visualización limpia de anotaciones originales

In [ ]:
# ============================================================
# VISUALIZACIÓN LIMPIA DE ANOTACIONES ORIGINALES
# ============================================================
# Objetivo:
# - evitar texto superpuesto sobre el ticket;
# - usar cajas verde oscuro;
# - identificar cada campo con un número;
# - mostrar la descripción en una leyenda lateral.

def _recortar_texto(texto, max_chars=38):
    """
    Recorta texto largo para que no se desborde en la leyenda.
    """
    texto = str(texto)
    return texto if len(texto) <= max_chars else texto[:max_chars - 3] + "..."


def _dibujar_numero(canvas, numero, x, y, color, radio=10):
    """
    Dibuja un círculo numerado pequeño.
    """
    cv2.circle(canvas, (int(x), int(y)), radio, color, -1)

    texto = str(numero)
    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.36, 1)

    cv2.putText(
        canvas,
        texto,
        (int(x - tw // 2), int(y + th // 2)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.36,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


def dibujar_cajas_numeradas(
    image_rgb,
    items,
    titulo_leyenda="Referencias",
    color_box=(0, 70, 0),
    margen_derecho=360,
    incluir_confianza=False
):
    """
    Dibuja cajas numeradas sobre una imagen RGB.

    items debe ser una lista de diccionarios con:
    - xmin, ymin, xmax, ymax
    - label
    - conf opcional
    """
    h, w = image_rgb.shape[:2]

    canvas = cv2.copyMakeBorder(
        image_rgb.copy(),
        top=0,
        bottom=0,
        left=0,
        right=margen_derecho,
        borderType=cv2.BORDER_CONSTANT,
        value=(255, 255, 255)
    )

    if not items:
        cv2.putText(
            canvas,
            "Sin elementos para mostrar",
            (w + 20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color_box,
            2,
            cv2.LINE_AA
        )
        return canvas

    # Orden visual: arriba-abajo, izquierda-derecha.
    items = sorted(items, key=lambda x: (x["ymin"], x["xmin"]))

    # Dibujar cajas y números sobre el ticket.
    for idx, item in enumerate(items, start=1):
        xmin = int(item["xmin"])
        ymin = int(item["ymin"])
        xmax = int(item["xmax"])
        ymax = int(item["ymax"])

        cv2.rectangle(canvas, (xmin, ymin), (xmax, ymax), color_box, 2)

        # Número a la izquierda del campo.
        cx = max(xmin - 16, 12)
        cy = int((ymin + ymax) / 2)
        cy = max(12, min(cy, h - 12))

        _dibujar_numero(canvas, idx, cx, cy, color_box, radio=10)

    # Leyenda lateral.
    legend_x = w + 20
    legend_y = 35

    cv2.putText(
        canvas,
        titulo_leyenda,
        (legend_x, legend_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        color_box,
        2,
        cv2.LINE_AA
    )

    start_y = legend_y + 30
    line_gap = 26

    for idx, item in enumerate(items, start=1):
        y = start_y + (idx - 1) * line_gap

        if y > h - 15:
            cv2.putText(
                canvas,
                "...",
                (legend_x, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (40, 40, 40),
                1,
                cv2.LINE_AA
            )
            break

        _dibujar_numero(canvas, idx, legend_x + 10, y - 5, color_box, radio=9)

        label = _recortar_texto(item["label"], max_chars=32)

        if incluir_confianza and "conf" in item:
            texto = f"{idx}. {label} ({item['conf']:.2f})"
        else:
            texto = f"{idx}. {label}"

        cv2.putText(
            canvas,
            texto,
            (legend_x + 28, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (40, 40, 40),
            1,
            cv2.LINE_AA
        )

    return canvas


def convertir_anotaciones_a_items(anns):
    """
    Convierte las anotaciones reales del DataFrame al formato común de dibujo.
    """
    items = []

    for _, row in anns.iterrows():
        items.append({
            "xmin": int(row["xmin"]),
            "ymin": int(row["ymin"]),
            "xmax": int(row["xmax"]),
            "ymax": int(row["ymax"]),
            "label": str(row["class"])
        })

    return items


def dibujar_anotaciones_originales(image_rgb, anns, box_color=(0, 70, 0)):
    """
    Dibuja anotaciones reales con estilo limpio y numerado.
    """
    items = convertir_anotaciones_a_items(anns)

    return dibujar_cajas_numeradas(
        image_rgb=image_rgb,
        items=items,
        titulo_leyenda="Anotaciones reales",
        color_box=box_color,
        margen_derecho=360,
        incluir_confianza=False
    )


def mostrar_imagen_con_anotaciones(split: str, filename: str, guardar: bool = False):
    """
    Muestra una imagen con sus anotaciones reales usando numeración y leyenda lateral.
    """
    img_path = DATASET_ROOT / split / filename
    image_bgr = cv2.imread(str(img_path))

    if image_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    anns = df[(df["split"] == split) & (df["filename"] == filename)]

    img_draw = dibujar_anotaciones_originales(
        image_rgb=image_rgb,
        anns=anns,
        box_color=(0, 70, 0)
    )

    titulo_corto = Path(filename).name
    if len(titulo_corto) > 90:
        titulo_corto = titulo_corto[:90] + "..."

    plt.figure(figsize=(15, 11))
    plt.imshow(img_draw)
    plt.axis("off")
    plt.title(f"Anotaciones originales - {split} / {titulo_corto}", fontsize=12)
    plt.tight_layout()

    if guardar:
        out_path = EVIDENCE_DIR / f"anotacion_original_limpia_{split}_{Path(filename).stem}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        print("Evidencia guardada en:", out_path)

    plt.show()


# Mostrar un ejemplo de train.
ejemplo = df[df["split"] == "train"].iloc[0]
mostrar_imagen_con_anotaciones("train", ejemplo["filename"], guardar=True)

## 10. Conversión de formato RetinaNet a YOLO

In [ ]:
def convertir_bbox_a_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    '''
    Convierte coordenadas absolutas xmin, ymin, xmax, ymax
    a formato YOLO normalizado:
    class_id x_center y_center width height
    '''
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h

    return x_center, y_center, width, height


def limpiar_y_crear_estructura_yolo():
    if YOLO_DIR.exists():
        shutil.rmtree(YOLO_DIR)

    for split in SPLITS:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)


def convertir_dataset_a_yolo():
    limpiar_y_crear_estructura_yolo()

    registros_conversion = []

    for split in SPLITS:
        split_dir = DATASET_ROOT / split
        imagenes = listar_imagenes(split_dir)

        df_split = df[df["split"] == split].copy()

        for img_path in imagenes:
            filename = img_path.name
            image = cv2.imread(str(img_path))

            if image is None:
                registros_conversion.append({
                    "split": split,
                    "filename": filename,
                    "estado": "imagen_no_leida",
                    "labels": 0
                })
                continue

            img_h, img_w = image.shape[:2]

            # Copiar imagen
            destino_img = YOLO_DIR / "images" / split / filename
            shutil.copy2(img_path, destino_img)

            anns = df_split[df_split["filename"] == filename]
            label_path = YOLO_DIR / "labels" / split / f"{img_path.stem}.txt"

            lines = []

            for _, row in anns.iterrows():
                clase = row["class"]

                if clase not in CLASS_TO_ID:
                    continue

                xmin = float(row["xmin"])
                ymin = float(row["ymin"])
                xmax = float(row["xmax"])
                ymax = float(row["ymax"])

                # Ajuste defensivo de coordenadas a límites de imagen
                xmin = max(0, min(xmin, img_w - 1))
                xmax = max(0, min(xmax, img_w - 1))
                ymin = max(0, min(ymin, img_h - 1))
                ymax = max(0, min(ymax, img_h - 1))

                if xmax <= xmin or ymax <= ymin:
                    continue

                x_center, y_center, width, height = convertir_bbox_a_yolo(
                    xmin, ymin, xmax, ymax, img_w, img_h
                )

                # Evitar valores fuera de rango por errores mínimos de redondeo
                valores = [x_center, y_center, width, height]
                if not all(0 <= v <= 1 for v in valores):
                    continue

                class_id = CLASS_TO_ID[clase]
                lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

            label_path.write_text("\n".join(lines), encoding="utf-8")

            registros_conversion.append({
                "split": split,
                "filename": filename,
                "estado": "convertida",
                "labels": len(lines)
            })

    return pd.DataFrame(registros_conversion)


conversion_df = convertir_dataset_a_yolo()

display(conversion_df.head())
display(conversion_df.groupby(["split", "estado"])["filename"].count().reset_index(name="cantidad"))
display(conversion_df.groupby("split")["labels"].sum().reset_index(name="labels_yolo"))

print("Dataset convertido a YOLO en:", YOLO_DIR)

## 11. Validación de archivos YOLO generados

In [ ]:
validacion_yolo = []

for split in SPLITS:
    images_dir = YOLO_DIR / "images" / split
    labels_dir = YOLO_DIR / "labels" / split

    imagenes = listar_imagenes(images_dir)
    labels = list(labels_dir.glob("*.txt"))

    validacion_yolo.append({
        "split": split,
        "imagenes_yolo": len(imagenes),
        "labels_yolo": len(labels),
        "coinciden": len(imagenes) == len(labels)
    })

validacion_yolo_df = pd.DataFrame(validacion_yolo)
display(validacion_yolo_df)

if not validacion_yolo_df["coinciden"].all():
    print("Atención: no coincide la cantidad de imágenes y labels en alguna partición.")
else:
    print("Validación correcta: cada imagen tiene su archivo .txt correspondiente.")

## 12. Técnicas de procesamiento de imágenes aplicadas al dataset

Esta sección se incorpora para demostrar el uso de técnicas clásicas de procesamiento de imágenes antes del entrenamiento del modelo.

Aunque el entrenamiento final de YOLOv8 utiliza las imágenes originales y sus anotaciones, estas transformaciones permiten analizar visualmente la estructura de los tickets hospitalarios y comprender mejor sus características.

Se aplican las siguientes técnicas:

- conversión a escala de grises;
- suavizado con Gaussian Blur;
- binarización;
- detección de bordes con Canny Edge Detection.

Estas técnicas son útiles para observar zonas de texto, bordes del documento, contraste y posibles dificultades visuales que podrían afectar la detección automática.

In [ ]:
# ============================================================
# 13.1 SELECCIÓN DE IMÁGENES DE MUESTRA PARA PROCESAMIENTO
# ============================================================

def obtener_imagenes_muestra(split="train", cantidad=3):
    '''
    Selecciona imágenes de muestra desde una partición del dataset.
    Se utilizan para aplicar técnicas clásicas de procesamiento de imágenes.
    '''
    split_dir = DATASET_ROOT / split

    imagenes = []
    for ext in IMAGE_EXTENSIONS:
        imagenes.extend(split_dir.glob(f"*{ext}"))
        imagenes.extend(split_dir.glob(f"*{ext.upper()}"))

    imagenes = sorted(set(imagenes))

    if not imagenes:
        raise FileNotFoundError(f"No se encontraron imágenes en {split_dir}")

    return imagenes[:cantidad]


imagenes_muestra = obtener_imagenes_muestra(split="train", cantidad=3)

print("Imágenes seleccionadas para procesamiento:")
for img in imagenes_muestra:
    print("-", img.name)

In [ ]:
# ============================================================
# 13.2 FUNCIÓN DE PROCESAMIENTO DE IMÁGENES
# ============================================================

def aplicar_tecnicas_procesamiento(img_path):
    '''
    Aplica técnicas de procesamiento de imágenes sobre una imagen:
    - escala de grises;
    - Gaussian Blur;
    - binarización;
    - Canny Edge Detection.
    '''

    image_bgr = cv2.imread(str(img_path))

    if image_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")

    # OpenCV lee las imágenes en BGR. Se convierte a RGB para mostrar correctamente.
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # 1. Escala de grises
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian Blur
    # Reduce ruido y suaviza variaciones menores antes de detectar bordes.
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # 3. Binarización con Otsu
    # Convierte la imagen a blanco y negro según un umbral calculado automáticamente.
    _, binary = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # 4. Canny Edge Detection
    # Detecta bordes relevantes del documento y de las zonas con texto.
    edges = cv2.Canny(blur, 50, 150)

    return {
        "original": image_rgb,
        "grises": gray,
        "gaussian_blur": blur,
        "binarizacion": binary,
        "canny": edges
    }

In [ ]:
# ============================================================
# 13.3 VISUALIZACIÓN DE TÉCNICAS SOBRE 2 O 3 IMÁGENES
# ============================================================

for img_path in imagenes_muestra:
    procesadas = aplicar_tecnicas_procesamiento(img_path)

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))

    axes[0].imshow(procesadas["original"])
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(procesadas["grises"], cmap="gray")
    axes[1].set_title("Escala de grises")
    axes[1].axis("off")

    axes[2].imshow(procesadas["gaussian_blur"], cmap="gray")
    axes[2].set_title("Gaussian Blur")
    axes[2].axis("off")

    axes[3].imshow(procesadas["binarizacion"], cmap="gray")
    axes[3].set_title("Binarización")
    axes[3].axis("off")

    axes[4].imshow(procesadas["canny"], cmap="gray")
    axes[4].set_title("Canny Edge Detection")
    axes[4].axis("off")

    plt.suptitle(f"Técnicas de procesamiento aplicadas a: {img_path.name}", fontsize=12)
    plt.tight_layout()

    out_path = EDA_DIR / f"procesamiento_imagen_{Path(img_path).stem}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")

    print("Evidencia guardada en:", out_path)

    plt.show()

### Interpretación de las técnicas aplicadas

La escala de grises simplifica la imagen y permite analizar la intensidad de los píxeles sin considerar color. En tickets y documentos administrativos, esto es útil porque la información principal suele estar dada por contraste entre texto y fondo.

Gaussian Blur suaviza la imagen y reduce pequeñas variaciones o ruido visual. Esto puede ayudar en etapas previas a la detección de bordes.

La binarización separa zonas claras y oscuras. En documentos, permite observar con mayor claridad áreas de texto, líneas y campos administrativos.

Canny Edge Detection resalta bordes y cambios fuertes de intensidad. En este proyecto permite identificar contornos del ticket y regiones donde hay texto o separadores.

Estas técnicas no reemplazan a YOLOv8, pero demuestran el análisis previo de las imágenes y ayudan a justificar que el dataset corresponde a un problema de procesamiento de imágenes documentales.

## 13. Creación del archivo data.yaml

In [ ]:
data_yaml = {
    "path": str(YOLO_DIR.resolve()),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

DATA_YAML_PATH = YOLO_DIR / "data.yaml"

with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("Archivo data.yaml creado en:", DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text(encoding="utf-8"))

## 14. Entrenamiento del modelo YOLOv8

In [ ]:
# ============================================================
# PARÁMETROS DE ENTRENAMIENTO
# ============================================================
# Para una prueba rápida: EPOCHS = 10
# Para una entrega final razonable: EPOCHS = 25
# Si tu PC no soporta IMG_SIZE = 960, probar 640

MODEL_BASE = "yolov8n.pt"
EPOCHS = 25
IMG_SIZE = 960
BATCH_SIZE = 2
CONF_THRESHOLD = 0.25

RUN_NAME = "tickets_hospitalarios_yolov8n_25ep"

model = YOLO(MODEL_BASE)

train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(OUTPUT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    device=DEVICE,
    workers=0,
    patience=7
)

print("Entrenamiento finalizado")

## 15. Lectura de resultados de entrenamiento

In [ ]:
TRAIN_RUN_DIR = OUTPUT_DIR / RUN_NAME
BEST_MODEL_PATH = TRAIN_RUN_DIR / "weights" / "best.pt"
LAST_MODEL_PATH = TRAIN_RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = TRAIN_RUN_DIR / "results.csv"

print("Ruta del mejor modelo:", BEST_MODEL_PATH)
print("Existe best.pt:", BEST_MODEL_PATH.exists())

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    display(results_df.tail())

    # Graficar pérdidas principales si existen
    columnas_loss = [c for c in results_df.columns if "loss" in c.lower()]

    for col in columnas_loss:
        plt.figure(figsize=(7, 4))
        plt.plot(results_df[col])
        plt.title(f"Evolución de {col}")
        plt.xlabel("Época")
        plt.ylabel(col)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        safe_col = col.replace("/", "_").replace(" ", "_")
        plt.savefig(EDA_DIR / f"training_{safe_col}.png", dpi=150)
        plt.show()
else:
    print("No se encontró results.csv. Revisar si el entrenamiento terminó correctamente.")

## 16. Evaluación del modelo sobre test

In [ ]:
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        "No se encontró best.pt. Ejecutá primero la celda de entrenamiento."
    )

best_model = YOLO(str(BEST_MODEL_PATH))

metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=IMG_SIZE,
    project=str(OUTPUT_DIR),
    name="evaluacion_test",
    exist_ok=True,
    device=DEVICE
)

print("Evaluación sobre test finalizada")
print(metrics)

## 17. Interpretación de métricas del modelo

Además de ejecutar la evaluación con YOLOv8, es importante interpretar las métricas en lenguaje humano.

En detección de objetos no alcanza con usar `accuracy`, porque el modelo debe acertar dos cosas al mismo tiempo:

1. qué clase detecta;
2. dónde ubica la caja o bounding box.

Por eso se utilizan métricas específicas como `Precision`, `Recall`, `mAP50` y `mAP50-95`.

In [ ]:
# ============================================================
# 17.1 RESUMEN DE MÉTRICAS DEL MODELO
# ============================================================

# Las métricas pueden variar según la versión de ultralytics.
# Este bloque intenta extraerlas de forma robusta.

metricas_resumen = {}

try:
    metricas_resumen["precision"] = float(metrics.box.mp)
    metricas_resumen["recall"] = float(metrics.box.mr)
    metricas_resumen["map50"] = float(metrics.box.map50)
    metricas_resumen["map50_95"] = float(metrics.box.map)
except Exception as e:
    print("No se pudieron extraer automáticamente todas las métricas.")
    print("Revisar el objeto metrics impreso en la celda anterior.")
    print("Detalle:", e)

if metricas_resumen:
    metricas_df = pd.DataFrame([
        {
            "metrica": "Precision",
            "valor": round(metricas_resumen["precision"], 4),
            "interpretacion": "De todas las detecciones realizadas, qué proporción fue correcta."
        },
        {
            "metrica": "Recall",
            "valor": round(metricas_resumen["recall"], 4),
            "interpretacion": "De todos los objetos reales, qué proporción logró detectar el modelo."
        },
        {
            "metrica": "mAP50",
            "valor": round(metricas_resumen["map50"], 4),
            "interpretacion": "Precisión media usando IoU mínimo de 0.50 entre caja real y caja predicha."
        },
        {
            "metrica": "mAP50-95",
            "valor": round(metricas_resumen["map50_95"], 4),
            "interpretacion": "Métrica más exigente; evalúa la precisión media con varios umbrales de IoU."
        },
    ])

    display(metricas_df)

    plt.figure(figsize=(8, 5))
    plt.bar(metricas_df["metrica"], metricas_df["valor"])

    plt.title("Resumen de métricas del modelo YOLOv8")
    plt.xlabel("Métrica")
    plt.ylabel("Valor")
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)

    for i, row in metricas_df.iterrows():
        plt.text(
            i,
            row["valor"],
            row["valor"],
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.savefig(EDA_DIR / "metricas_modelo_yolov8.png", dpi=150)
    plt.show()

### Explicación de métricas en lenguaje humano

**Precision** indica qué tan confiables son las detecciones que realiza el modelo. Si la precisión es alta, significa que cuando el modelo marca un campo en el ticket, generalmente lo marca correctamente.

**Recall** indica qué tan bien el modelo encuentra los campos reales. Si el recall es alto, significa que el modelo omite pocos campos relevantes.

**mAP50** mide la precisión media de detección considerando que una predicción es correcta si su caja se superpone suficientemente con la caja real usando un umbral IoU de 0.50.

**mAP50-95** es más exigente porque evalúa varios umbrales de superposición entre 0.50 y 0.95. Esta métrica suele ser menor que mAP50 y permite analizar con mayor rigor la calidad de localización de las cajas.

En este proyecto, estas métricas permiten evaluar si YOLOv8 detecta correctamente campos administrativos como paciente, factura, autorización, código, cantidad, copago, cuota moderadora, total, ingreso y valor unitario.

## 18. Predicción sobre imágenes de prueba

In [ ]:
# ============================================================
# PREDICCIÓN SOBRE IMÁGENES DE TEST
# ============================================================
# Se ejecutan predicciones sobre el conjunto de prueba.
# No se utiliza save=True para evitar guardar imágenes con el render default de YOLO,
# ya que luego se genera una visualización limpia con numeración y leyenda lateral.

TEST_IMAGES_DIR = YOLO_DIR / "images" / "test"

test_images = []
for ext in IMAGE_EXTENSIONS:
    test_images.extend(TEST_IMAGES_DIR.glob(f"*{ext}"))
    test_images.extend(TEST_IMAGES_DIR.glob(f"*{ext.upper()}"))

test_images = sorted(set(test_images))

if not test_images:
    raise FileNotFoundError(f"No se encontraron imágenes de test en: {TEST_IMAGES_DIR}")

pred_results = best_model.predict(
    source=str(TEST_IMAGES_DIR),
    imgsz=IMG_SIZE,
    conf=CONF_THRESHOLD,
    save=False,
    verbose=False,
    device=DEVICE
)

print("Predicciones generadas en memoria.")
print("Cantidad de imágenes evaluadas:", len(test_images))

## 19. Visualización limpia de predicciones

In [ ]:
# ============================================================
# VISUALIZACIÓN LIMPIA DE PREDICCIONES
# ============================================================

def convertir_predicciones_a_items(resultado):
    """
    Convierte las predicciones YOLO al formato común de dibujo.
    """
    items = []

    boxes = resultado.boxes

    if boxes is None or len(boxes) == 0:
        return items

    for i in range(len(boxes)):
        xyxy = boxes.xyxy[i].cpu().numpy()
        cls_id = int(boxes.cls[i].cpu().numpy())
        conf = float(boxes.conf[i].cpu().numpy())

        xmin, ymin, xmax, ymax = map(int, xyxy)

        items.append({
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "label": resultado.names[cls_id],
            "conf": conf
        })

    return items


def dibujar_prediccion_numerada(image_rgb, resultado, color_box=(0, 70, 0)):
    """
    Dibuja predicciones del modelo con estilo limpio:
    - cajas verde oscuro;
    - número a la izquierda del campo;
    - leyenda lateral con clase y confianza.
    """
    items = convertir_predicciones_a_items(resultado)

    return dibujar_cajas_numeradas(
        image_rgb=image_rgb,
        items=items,
        titulo_leyenda="Predicciones",
        color_box=color_box,
        margen_derecho=380,
        incluir_confianza=True
    )


def mostrar_prediccion_limpia(indice: int = 0, split: str = "test", conf: float = 0.25):
    """
    Muestra una predicción limpia usando la imagen original.
    Evita el formato default de YOLO, que suele solapar etiquetas sobre el ticket.
    """
    img_dir = YOLO_DIR / "images" / split

    imagenes = []
    for ext in IMAGE_EXTENSIONS:
        imagenes.extend(img_dir.glob(f"*{ext}"))
        imagenes.extend(img_dir.glob(f"*{ext.upper()}"))

    imagenes = sorted(set(imagenes))

    if not imagenes:
        raise FileNotFoundError(f"No se encontraron imágenes en: {img_dir}")

    indice = max(0, min(indice, len(imagenes) - 1))
    img_path = imagenes[indice]

    print("Imagen seleccionada:", img_path.name)

    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    resultado = best_model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=conf,
        save=False,
        verbose=False,
        device=DEVICE
    )[0]

    print("Cantidad de detecciones:", len(resultado.boxes) if resultado.boxes is not None else 0)

    img_draw = dibujar_prediccion_numerada(
        image_rgb=image_rgb,
        resultado=resultado,
        color_box=(0, 70, 0)
    )

    titulo_corto = img_path.name
    if len(titulo_corto) > 90:
        titulo_corto = titulo_corto[:90] + "..."

    plt.figure(figsize=(15, 11))
    plt.imshow(img_draw)
    plt.axis("off")
    plt.title(f"Predicción limpia - {split} / {titulo_corto}", fontsize=12)
    plt.tight_layout()

    out_path = EVIDENCE_DIR / f"prediccion_limpia_{split}_{Path(img_path).stem}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print("Evidencia guardada en:", out_path)

    plt.show()


mostrar_prediccion_limpia(indice=0, split="test", conf=CONF_THRESHOLD)

## 20. Comparación visual: anotación real vs predicción del modelo

In [ ]:
# ============================================================
# COMPARACIÓN VISUAL LIMPIA: ANOTACIÓN REAL VS PREDICCIÓN
# ============================================================

def comparar_anotacion_vs_prediccion(split: str = "test", indice: int = 0, conf: float = 0.25):
    """
    Compara visualmente:
    - anotaciones reales del dataset;
    - predicciones generadas por YOLOv8.

    Ambas visualizaciones usan el mismo criterio:
    cajas verde oscuro, números identificadores y leyenda lateral.
    """
    df_split = df[df["split"] == split].copy()
    filenames = sorted(df_split["filename"].unique())

    if not filenames:
        print(f"No hay imágenes anotadas en split={split}")
        return

    indice = max(0, min(indice, len(filenames) - 1))
    filename = filenames[indice]

    img_path = DATASET_ROOT / split / filename
    image_bgr = cv2.imread(str(img_path))

    if image_bgr is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {img_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Anotación real.
    anns = df[(df["split"] == split) & (df["filename"] == filename)]
    img_real = dibujar_anotaciones_originales(
        image_rgb=image_rgb,
        anns=anns,
        box_color=(0, 70, 0)
    )

    # Predicción.
    resultado = best_model.predict(
        source=str(img_path),
        imgsz=IMG_SIZE,
        conf=conf,
        save=False,
        verbose=False,
        device=DEVICE
    )[0]

    img_pred = dibujar_prediccion_numerada(
        image_rgb=image_rgb,
        resultado=resultado,
        color_box=(0, 70, 0)
    )

    print("Imagen comparada:", filename)
    print("Anotaciones reales:", len(anns))
    print("Predicciones detectadas:", len(resultado.boxes) if resultado.boxes is not None else 0)

    fig, axes = plt.subplots(1, 2, figsize=(22, 10))

    axes[0].imshow(img_real)
    axes[0].axis("off")
    axes[0].set_title("Anotación real", fontsize=12)

    axes[1].imshow(img_pred)
    axes[1].axis("off")
    axes[1].set_title("Predicción del modelo", fontsize=12)

    plt.tight_layout()

    out_path = EVIDENCE_DIR / f"comparacion_limpia_real_vs_pred_{split}_{Path(filename).stem}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print("Imagen comparativa guardada en:", out_path)

    plt.show()


comparar_anotacion_vs_prediccion(split="test", indice=0, conf=CONF_THRESHOLD)

## 21. Exportación de resumen para informe

In [ ]:
resumen_proyecto = {
    "nombre_proyecto": "Detector Inteligente de Campos en Tickets Hospitalarios",
    "enfoque": "Detección de objetos en imágenes de tickets hospitalarios",
    "modelo": MODEL_BASE,
    "epochs": EPOCHS,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "clases": CLASS_NAMES,
    "imagenes_train": int(len(listar_imagenes(DATASET_ROOT / "train"))),
    "imagenes_valid": int(len(listar_imagenes(DATASET_ROOT / "valid"))),
    "imagenes_test": int(len(listar_imagenes(DATASET_ROOT / "test"))),
    "anotaciones_totales": int(len(df)),
    "imagenes_unicas_anotadas": int(df["filename"].nunique()),
    "dataset_root": str(DATASET_ROOT),
    "yolo_dataset": str(YOLO_DIR),
    "best_model": str(BEST_MODEL_PATH),
}

resumen_path = OUTPUT_DIR / "resumen_proyecto.json"
with open(resumen_path, "w", encoding="utf-8") as f:
    json.dump(resumen_proyecto, f, ensure_ascii=False, indent=4)

txt_path = OUTPUT_DIR / "resumen_proyecto.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("RESUMEN DEL PROYECTO\n")
    f.write("====================\n\n")
    for k, v in resumen_proyecto.items():
        f.write(f"{k}: {v}\n")

print("Resumen JSON:", resumen_path)
print("Resumen TXT:", txt_path)
print(txt_path.read_text(encoding="utf-8"))

## 22. Conclusión técnica para usar en la entrega

In [ ]:
conclusion = '''
El proyecto implementa un modelo de detección de objetos para localizar campos administrativos dentro de tickets hospitalarios.
El dataset fue validado, analizado exploratoriamente y convertido desde formato RetinaNet a formato YOLO.
Luego se entrenó un modelo YOLOv8, adecuado para detección de regiones dentro de imágenes documentales.

El enfoque se diferencia de una clasificación tradicional porque no asigna una única categoría a toda la imagen, sino que identifica múltiples campos dentro del mismo ticket.
Esto permite avanzar hacia una futura automatización de carga o validación documental en contextos administrativos de salud.

Las principales limitaciones posibles son la cantidad de imágenes disponibles, el balance entre clases, la calidad de las anotaciones y la variabilidad visual de los tickets.
Como mejora futura, se podría integrar OCR sobre las regiones detectadas para extraer el texto de cada campo localizado.
'''

print(conclusion)

## 23. Limitaciones y mejoras futuras

### Limitaciones

El modelo puede presentar dificultades en los siguientes casos:

- tickets con baja calidad de impresión;
- imágenes borrosas o mal enfocadas;
- campos muy pequeños;
- superposición entre campos;
- variaciones grandes en el formato del ticket;
- diferencias de iluminación, rotación o recorte;
- baja cantidad de ejemplos en comparación con un sistema productivo real.

Estas limitaciones son esperables en problemas de visión computacional documental, especialmente cuando el dataset es acotado.

### Mejoras futuras

Como líneas futuras de mejora se proponen:

- entrenar con mayor cantidad de imágenes;
- incorporar tickets con más variaciones visuales;
- aplicar aumento de datos;
- integrar OCR sobre las regiones detectadas;
- validar automáticamente los textos extraídos;
- conectar el modelo con un sistema administrativo hospitalario;
- comparar YOLOv8 con otros modelos de detección;
- implementar una interfaz simple para cargar tickets y visualizar campos detectados.

### Conclusión ampliada

El proyecto cumple con un flujo completo de procesamiento de imágenes y aprendizaje profundo. Primero se analiza el dataset, luego se aplican técnicas clásicas de procesamiento visual, posteriormente se convierte el dataset a formato YOLO y finalmente se entrena y evalúa un modelo de detección de objetos.

El enfoque es adecuado porque cada ticket contiene múltiples campos administrativos. Por eso, usar YOLOv8 es más pertinente que una CNN clasificadora tradicional, ya que el objetivo no es clasificar la imagen completa, sino localizar regiones específicas dentro del documento.